In [32]:
from pathlib import Path
import utils as ut
import matplotlib.pyplot as plt
import matplotlib

import pandas as pd
pd.set_option('display.max_columns', None)

from astropy.io import fits
from astropy.nddata import Cutout2D
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.wcs import WCS

In [33]:
cwd = Path.cwd()
table_dir = cwd / 'data' / 'tables'
mosaic_dir = cwd / 'data' / 'mosaics'
cutout_dir = cwd / 'data' / 'cutouts'

In [34]:
table_dfs = {}
for file in table_dir.iterdir():
    table_dfs[file.stem] = pd.read_csv(file).sort_values(by="RA")

headers = ut.fits_df(mosaic_dir)
headers = headers[['FILENAME'] + [col for col in headers.columns if col != 'FILENAME']]

In [35]:
size = u.Quantity([3, 3], u.arcsec)

for idx, row in headers.iterrows():
    path = row['FILEPATH']
    catalogs = [('ceers', 'ceers'),
                ('cosmos', 'primer_cosmos'),
                ('uds', 'primer_uds')]
    bands = ['f150w', 'f277w', 'f444w']

    band = None
    catalog = None

    for cat, b in zip(catalogs, bands):
        if cat[0] in path.name:
            catalog = cat[1]
        if b in path.name:
            band = b

    table = table_dfs[catalog]

    hdul = fits.open(path)
    data = hdul[0].data
    wcs = WCS(hdul[0].header)

    for idx, row in table.iterrows():
        pos = SkyCoord(row['RA'], row['DEC'], frame="icrs", unit=u.deg)
        cutout = Cutout2D(data, pos, size, wcs=wcs)

        header = cutout.wcs.to_header()
        for col_name, value in row.items():
            if pd.isna(value):
                continue
            key = str(col_name)[:8].upper()
            header[key] = value
        header['BAND'] = band
        header['CATALOG'] = catalog

        folder = Path(cutout_dir / f"{row['id']}")
        folder.mkdir(parents=True, exist_ok=True)

        cutout_fits = fits.PrimaryHDU(data=cutout.data, header=header)
        cutout_fits.writeto(folder / f"{band}.fits", overwrite=True)

    hdul.close()


Set DATE-AVG to '2022-09-21T10:17:11.746' from MJD-AVG.
Set DATE-END to '2022-12-22T05:42:36.326' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -36.758463 from OBSGEO-[XYZ].
Set OBSGEO-H to 1725319218.494 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2022-09-22T03:18:18.036' from MJD-AVG.
Set DATE-END to '2022-12-25T03:57:18.887' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -36.739053 from OBSGEO-[XYZ].
Set OBSGEO-H to 1725216481.136 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2022-10-07T06:02:04.794' from MJD-AVG.
Set DATE-END to '2022-12-22T06:44:09.762' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -36.768884 from OBSGEO-[XYZ].
Set OBSGEO-H to 1725373838.960 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


In [36]:
size = len(list(cutout_dir.glob('*')))
print(f"Number of cutouts: {size}")

Number of cutouts: 486


In [ ]:
for folder in cutout_dir.iterdir():
    if folder.is_dir():
        count = len(list(folder.glob('*')))
        if count != 3:
            print(f"{folder.name}")